# ImageNet evaluation with canonical label fix + TransAxx

This notebook fixes the numeric-folder ImageNet label problem before running any approximate-multiplier experiments.

Correct order:
1. Load the clean pretrained ViT model.
2. Run the **recover labels** cell once. It creates `folder_to_timm.npy` and should print corrected top-1 around 80%.
3. Run the **devkit diff** cell. It uses the included devkit metadata and creates `folder_to_timm_canonical.npy`.
4. Use the corrected `ImageFolder(..., target_transform=...)` loaders for all final baseline and AXX numbers.

Do **not** use `NumericImageFolder` anymore for final thesis numbers.


In [ ]:
# Run this once in JupyterLab to install packages into this notebook kernel.
import subprocess
import sys
from pathlib import Path

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/home/jovyan/transaxx'), Path('/workspace/Handover')]
    for candidate in candidates:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'classification').is_dir():
            return candidate.resolve()
    raise RuntimeError('Could not find repo root containing requirements.txt and classification/.')

repo_root = find_repo_root()
requirements = repo_root / 'requirements.txt'
print('Python:', sys.executable)
print('Installing:', requirements)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])
print('Done. If imports still fail, restart the kernel and rerun the setup cell.')


In [ ]:
import os
import sys
import time
import glob
from pathlib import Path
import numpy as np
import torch

def find_transaxx_root():
    # If needed, set this manually to the folder that contains classification/.
    manual_path = os.environ.get("TRANSAXX_PATH", "").strip()
    candidates = []
    if manual_path:
        candidates.append(Path(manual_path).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    candidates.extend([Path("/home/jovyan/transaxx"), Path("/workspace/Handover")])

    for candidate in candidates:
        if (candidate / "classification").is_dir() and (candidate / "pytorch-quantization").is_dir():
            return candidate.resolve()

    raise RuntimeError(
        "Could not find the TransAxx repo root. Set TRANSAXX_PATH to the folder "
        "that directly contains classification/ and pytorch-quantization/."
    )

TRANSAXX_PATH = str(find_transaxx_root())
IMAGENET_PATH = os.path.join(TRANSAXX_PATH, "datasets", "imagenet_data1")
VAL_DIR = os.path.join(IMAGENET_PATH, "val")
CALIB_DIR = os.path.join(IMAGENET_PATH, "train_tiny")
DEVKIT_ROOT = os.path.join(TRANSAXX_PATH, "datasets")

os.environ["PYTHONPATH"] = TRANSAXX_PATH
sys.path.insert(0, TRANSAXX_PATH)
sys.path.insert(0, os.path.join(TRANSAXX_PATH, "pytorch-quantization"))
print("TRANSAXX_PATH:", TRANSAXX_PATH)

from classification.utils import *
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from timm.data import resolve_data_config, create_transform

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
print("VAL_DIR:", VAL_DIR)
print("CALIB_DIR:", CALIB_DIR)


## 1. Load the clean pretrained model

In [2]:
import timm

model_name = "vit_small_patch16_224"
model = timm.create_model(model_name, pretrained=True).to(device)
model.eval()

print("Model loaded:", model_name)


Model loaded: vit_small_patch16_224


## 2. Recover the folder → timm label map

Run this **before any approximation**.  
This uses the clean pretrained model as a diagnostic oracle. It creates `folder_to_timm.npy`.

Expected result: corrected top-1 around **80%**. That means preprocessing is fine and the original problem was label mapping.


In [3]:
# ============================================================================
# RECOVER THE VAL LABEL MAPPING (no devkit / no downloads needed)
# Run on the CLEAN pretrained model, before any approximation.
# Idea: the model is ~81% accurate & confident, so the class it predicts MOST
# often inside a folder IS that folder's true timm class. One pass over val
# recovers the folder -> timm-index map and verifies the corrected accuracy.
# ============================================================================
import torch, numpy as np
from tqdm import tqdm
from timm.data import resolve_data_config, create_transform
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# VAL_DIR is defined in the setup cell; edit it there if needed

model.eval()
cfg = resolve_data_config({}, model=model)
tf  = create_transform(**cfg, is_training=False)

# Any consistent folder ordering works here -- we recover the true map regardless.
ds     = ImageFolder(VAL_DIR, tf)
loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=16, pin_memory=True)
idx_to_folder = {v: k for k, v in ds.class_to_idx.items()}
print(f"folders: {len(ds.classes)}  (names e.g. {ds.classes[:3]} ... {ds.classes[-3:]})")

# --- one pass: record (folder_index, predicted_timm_index) for every image ---
folder_all, pred_all = [], []
with torch.no_grad():
    for imgs, fidx in tqdm(loader, total=len(loader)):
        pred = model(imgs.to(device)).argmax(1).cpu()
        pred_all.append(pred); folder_all.append(fidx)
pred_all   = torch.cat(pred_all).numpy()
folder_all = torch.cat(folder_all).numpy()

# --- per folder, the most-predicted timm class = that folder's true class ---
n_folders = len(ds.classes)
folder_to_timm = np.full(n_folders, -1, dtype=int)
for f in range(n_folders):
    preds_f = pred_all[folder_all == f]
    if len(preds_f) == 0:
        continue
    vals, counts = np.unique(preds_f, return_counts=True)
    folder_to_timm[f] = vals[counts.argmax()]

distinct = len(set(folder_to_timm.tolist()) - {-1})
corrected = 100 * (pred_all == folder_to_timm[folder_all]).mean()
print(f"\ndistinct recovered classes: {distinct} / {n_folders}  (1000 = clean bijection)")
print(f"corrected top-1 over all val: {corrected:.1f}%   (expect ~80%)")

# Save the map so you can reuse it without re-running this pass.
np.save("folder_to_timm.npy", folder_to_timm)
print("saved folder_to_timm.npy")

# ============================================================================
# HOW TO USE IT IN YOUR EVAL
# Rebuild the val dataset with a target_transform that relabels on the fly.
# (Same VAL_DIR + default ImageFolder sort => indices match folder_to_timm.)
# Then feed val_data to evaluate_imagenet exactly as before.
# ----------------------------------------------------------------------------
# f2t = torch.tensor(np.load("folder_to_timm.npy"))
# val_ds = ImageFolder(VAL_DIR, transform=tf, target_transform=lambda y: int(f2t[y]))
# val_data = DataLoader(val_ds, batch_size=128, shuffle=False,
#                       num_workers=16, pin_memory=True)
# ============================================================================


folders: 1000  (names e.g. ['0', '1', '10'] ... ['997', '998', '999'])


100%|██████████| 391/391 [06:11<00:00,  1.05it/s]


distinct recovered classes: 996 / 1000  (1000 = clean bijection)
corrected top-1 over all val: 81.5%   (expect ~80%)
saved folder_to_timm.npy


In [4]:
# Run this if scipy is missing.
# If your environment already has scipy, you can skip it.
%pip install scipy -q



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 3. Build the canonical devkit map

Run this after:
- `folder_to_timm.npy` exists from step 2
- `datasets/ILSVRC2012_devkit_t12/data/meta.mat` exists from the included devkit directory

It writes `folder_to_timm_canonical.npy`, which is the map to use for final thesis numbers.


In [6]:
# ============================================================================
# CANONICAL LABEL FIX via the ImageNet devkit  (the defensible, final-results map)
#
# Needs:
#   - datasets/ILSVRC2012_devkit_t12/data/meta.mat from the included devkit directory
#   - folder_to_timm.npy  from the recover-labels cell (used as a cross-check oracle)
#
# It builds the canonical folder -> timm-class map from the devkit, then DIFFS it
# against the model-recovered map to prove what your numeric folders actually are.
# Run on the clean pretrained model context (only needs numpy/scipy, no GPU).
# ============================================================================
import os, glob, numpy as np, scipy.io as sio
from torchvision.datasets import ImageFolder

# VAL_DIR and DEVKIT_ROOT are defined in the setup cell; edit them there if needed

# --- locate meta.mat ---
hits = glob.glob(os.path.join(DEVKIT_ROOT, "**", "meta.mat"), recursive=True)
assert hits, f"meta.mat not found under {DEVKIT_ROOT}. Expected datasets/ILSVRC2012_devkit_t12/data/meta.mat."
meta_path = hits[0]
print("using", meta_path)

# --- devkit -> canonical maps (mirrors torchvision's parse_meta_mat) ---
meta = sio.loadmat(meta_path, squeeze_me=True)["synsets"]
nums_children = list(zip(*meta))[4]                       # field 4 = num_children
leaves = [meta[i] for i, nc in enumerate(nums_children) if nc == 0]   # the 1000 classes
idcs, wnids = list(zip(*leaves))[:2]                      # ILSVRC2012_ID (1..1000), WNID
ilsvrc_id_to_wnid = {int(i): str(w) for i, w in zip(idcs, wnids)}
# timm/torch class index == lexicographic rank of the WNID among the 1000 leaf WNIDs
wnid_to_timm = {w: i for i, w in enumerate(sorted(ilsvrc_id_to_wnid.values()))}

# --- his folder names, in the SAME order folder_to_timm.npy was built ---
classes   = ImageFolder(VAL_DIR).classes        # default (string) sort; matches recover cell
recovered = np.load("folder_to_timm.npy")       # model-derived oracle, ~99%+ correct
assert len(classes) == len(recovered), "folder count changed since recover cell — re-run it"

# --- test what the folder numbers mean, using the oracle to disambiguate ---
def build(hyp):
    out = np.full(len(classes), -1, dtype=int)
    for i, name in enumerate(classes):
        try:
            k = int(name)
        except ValueError:
            return None                          # folders aren't integers -> different scheme
        if   hyp == "ilsvrc_1based":  ilsvrc_id = k          # folders "1".."1000"
        elif hyp == "ilsvrc_0based":  ilsvrc_id = k + 1      # folders "0".."999"
        elif hyp == "timm_direct":    out[i] = k; continue   # folders already == timm index
        out[i] = wnid_to_timm.get(ilsvrc_id_to_wnid.get(ilsvrc_id), -1)
    return out

best = None
for hyp in ("ilsvrc_1based", "ilsvrc_0based", "timm_direct"):
    cand = build(hyp)
    if cand is None:
        print("folders are not plain integers — ILSVRC-ID hypotheses don't apply"); break
    agree = 100 * (cand == recovered).mean()
    print(f"{hyp:>14s}: agrees with model-recovered map on {agree:.1f}% of folders")
    if best is None or agree > best[1]:
        best = (hyp, agree, cand)

hyp, agree, canonical = best
print(f"\nbest convention: {hyp}   ({agree:.1f}% agreement)")
if agree > 99:
    np.save("folder_to_timm_canonical.npy", canonical)
    print(">>> CLEAN: your folder names ARE devkit IDs. Canonical map confirmed + saved")
    print("    -> folder_to_timm_canonical.npy   (use THIS for your thesis numbers)")
    # where do the two maps disagree? (a few hard classes, expected)
    bad = np.where(canonical != recovered)[0]
    print(f"    {len(bad)} folders differ from the model-recovered map "
          f"(expected: hard classes the model mislabels by majority)")
else:
    print(">>> NO clean devkit interpretation of your folder numbers.")
    print("    Re-stage val from the ORIGINAL tar instead (guaranteed correct):")
    print("      from torchvision.datasets import ImageNet")
    print("      ImageNet('<dir holding ILSVRC2012_img_val.tar + the devkit tar>', split='val')")
    print("    then load val/<WNID>/ with a plain ImageFolder (no NumericImageFolder).")

# ============================================================================
# USE IT IN YOUR EVAL  (canonical map; same pattern as before)
#   f2t = torch.tensor(np.load("folder_to_timm_canonical.npy"))
#   val_ds = ImageFolder(VAL_DIR, transform=tf, target_transform=lambda y: int(f2t[y]))
#   val_data = DataLoader(val_ds, batch_size=128, shuffle=False,
#                         num_workers=16, pin_memory=True)
# ============================================================================


using /home/jovyan/transaxx/datasets/ILSVRC2012_devkit_t12/data/meta.mat
 ilsvrc_1based: agrees with model-recovered map on 0.1% of folders
 ilsvrc_0based: agrees with model-recovered map on 99.2% of folders
   timm_direct: agrees with model-recovered map on 0.0% of folders

best convention: ilsvrc_0based   (99.2% agreement)
>>> CLEAN: your folder names ARE devkit IDs. Canonical map confirmed + saved
    -> folder_to_timm_canonical.npy   (use THIS for your thesis numbers)
    8 folders differ from the model-recovered map (expected: hard classes the model mislabels by majority)


## 4. Corrected ImageNet loaders

This replaces the old `NumericImageFolder` approach.

Use `folder_to_timm_canonical.npy` for final numbers.  
For calibration-only stats, labels do not matter. For retraining, labels do matter, so `train_tiny` gets the same transform if its folder structure matches `val`.


In [7]:
def make_corrected_imagenet_loaders(
    imagenet_path=IMAGENET_PATH,
    map_path="folder_to_timm_canonical.npy",
    batch_size=128,
    num_workers=16,
):
    val_dir = os.path.join(imagenet_path, "val")
    calib_dir = os.path.join(imagenet_path, "train_tiny")

    assert os.path.exists(map_path), (
        f"{map_path} not found. Run the recover-labels and devkit-diff cells first."
    )

    cfg = resolve_data_config({}, model=model)
    tf = create_transform(**cfg, is_training=False)

    f2t_np = np.load(map_path)
    f2t = torch.tensor(f2t_np, dtype=torch.long)

    val_ds = ImageFolder(
        val_dir,
        transform=tf,
        target_transform=lambda y: int(f2t[y]),
    )

    assert len(val_ds.classes) == len(f2t_np), (
        f"Map length {len(f2t_np)} does not match val folder count {len(val_ds.classes)}. "
        "Re-run recover/devkit cells with the same VAL_DIR."
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    # train_tiny/calib: same folder labels if it has the same numeric folder scheme.
    if os.path.exists(calib_dir):
        calib_ds_plain = ImageFolder(calib_dir, transform=tf)
        if len(calib_ds_plain.classes) == len(f2t_np):
            calib_ds = ImageFolder(
                calib_dir,
                transform=tf,
                target_transform=lambda y: int(f2t[y]),
            )
            print("Calibration/retraining loader uses corrected labels.")
        else:
            calib_ds = calib_ds_plain
            print(
                "WARNING: train_tiny folder count differs from val. "
                "Using plain labels for calib_data. This is OK for quantization calibration, "
                "but NOT OK for supervised retraining unless labels are correct."
            )

        calib_loader = DataLoader(
            calib_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True,
        )
    else:
        calib_loader = None
        print("WARNING: CALIB_DIR not found:", calib_dir)

    return val_loader, calib_loader

val_data, calib_data = make_corrected_imagenet_loaders(batch_size=128)

print("Validation batches:", len(val_data))
print("Calibration batches:", len(calib_data) if calib_data is not None else None)


Calibration/retraining loader uses corrected labels.
Validation batches: 391
Calibration batches: 79


## 5. Check corrected FP32 baseline

In [8]:
criterion = torch.nn.CrossEntropyLoss().to(device)

model.eval()
top1, top5 = evaluate_imagenet(
    model,
    val_data,
    criterion,
    print_freq=1000,
    device=device,
)

print("Corrected FP32 baseline:")
print("Top-1:", top1)
print("Top-5:", top5)


Starting evaluation...


100%|██████████| 391/391 [07:04<00:00,  1.09s/it]

 * Acc@1 81.380 Acc@5 96.136
Corrected FP32 baseline:
Top-1: 81.38
Top-5: 96.136


## 6. Initialize TransAxx linear layers

From here onward you can run your approximate multiplier sweep.  
For clean comparisons, restart the kernel and rerun from the top before each new multiplier, or reload the clean pretrained model before replacing layers.


In [8]:
# get linear layers to approximate
linear_layers = [(name, module) for name, module in model.named_modules() if (isinstance(module, torch.nn.Linear) or  isinstance(module, AdaPT_Linear)) and ("head" not in name and "reduction" not in name)]

In [9]:
len(linear_layers)

48

In [10]:
# Initialize model with all required approximate multipliers for axx layers. 
# No explicit assignment needed; this step JIT compiles all upcoming multipliers

axx_list = [{'axx_mult' : 'mul8s_acc', 'axx_power' : 1.0, 'quant_bits' : 8, 'fake_quant' : False}]*len(linear_layers)

axx_list[1:2] = [{'axx_mult' : 'mul8s_1L2H', 'axx_power' : 0.8871, 'quant_bits' : 8, 'fake_quant' : False}] * 1

start = time.time()
replace_linear_layers(model,  AdaPT_Linear, axx_list, 0, 0, layer_count=[0], returned_power = [0], initialize = True)  
print('Time to compile cuda extensions: ', time.time()-start)

/usr/local/lib/python3.10/dist-packages/torch/utils/cpp_extension.py:1967: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/cpp_extension.py:1967: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


Time to compile cuda extensions:  101.11960792541504


In [11]:
# measure flops of model and compute 'flops' in every layer

import io
from classification.ptflops import get_model_complexity_info
from classification.ptflops.pytorch_ops import linear_flops_counter_hook
from classification.ptflops.pytorch_ops import conv_flops_counter_hook

#hook our custom axx_layers in the appropriate flop counters, i.e. AdaPT_Linear : linear_flops_counter_hook
with torch.cuda.device(0):
    total_macs, total_params, layer_specs = get_model_complexity_info(model, (3, 224, 224),as_strings=False, print_per_layer_stat=True,
                                                          custom_modules_hooks={AdaPT_Linear : linear_flops_counter_hook}, 
                                                          param_units='M', flops_units='MMac',
                                                          verbose=True)

print(f'Computational complexity:  {total_macs/1000000:.2f} MMacs')
print(f'Number of parameters::  {total_params/1000000:.2f} MParams')


Computational complexity:  4244.97 MMacs
Number of parameters::  22.05 MParams


## Run model calibration for quantization

Calibrates the quantization parameters 

Need to re-run it each time the initial model changes

In [12]:
with torch.no_grad():
    stats = collect_stats(model, calib_data, num_batches=2, device=device)
    amax = compute_amax(model, method="percentile", percentile=99.99, device=device)
    
    # optional - test different calibration methods
    #amax = compute_amax(model, method="mse")
    #amax = compute_amax(model, method="entropy")

100%|██████████| 2/2 [00:35<00:00, 17.65s/it]
W0627 21:26:21.198394 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.200329 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.201614 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.202581 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.203643 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.204655 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.205402 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.206120 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.206926 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.207653 140186354698048 tensor_quantizer.py:173] Disable HistogramCalibrator
W0627 21:26:21.208447 140186354698048 tensor

blocks.0.attn.qkv.quantizer             : TensorQuantizer(8bit per-tensor amax=4.5153 calibrator=HistogramCalibrator scale=33.77614212036133 quant)
blocks.0.attn.qkv.quantizer_w           : TensorQuantizer(8bit per-tensor amax=0.3660 calibrator=HistogramCalibrator scale=247.86000061035156 quant)
blocks.0.attn.proj.quantizer            : TensorQuantizer(8bit per-tensor amax=2.6404 calibrator=HistogramCalibrator scale=0.0050275190733373165 quant)
blocks.0.attn.proj.quantizer_w          : TensorQuantizer(8bit per-tensor amax=0.5768 calibrator=HistogramCalibrator scale=135.45745849609375 quant)
blocks.0.mlp.fc1.quantizer              : TensorQuantizer(8bit per-tensor amax=7.5785 calibrator=HistogramCalibrator scale=24.273090362548828 quant)
blocks.0.mlp.fc1.quantizer_w            : TensorQuantizer(8bit per-tensor amax=0.4483 calibrator=HistogramCalibrator scale=167.78167724609375 quant)
blocks.0.mlp.fc2.quantizer              : TensorQuantizer(8bit per-tensor amax=5.7716 calibrator=Histogr

## 7. Run approximate evaluation

Edit the multiplier name, `axx_power`, and layer range here.

Your current example approximates the first 47 linear layers with `mul8s_FPGA_ISH2`.


In [13]:
# set desired approximate multiplier in each layer

#at first, set all layers to have the 8-bit accurate multiplier
axx_list = [{'axx_mult' : 'mul8s_acc', 'axx_power' : 1.0, 'quant_bits' : 8, 'fake_quant' : False}]*len(linear_layers)

# For example, set the first 5 layers to be approximated with a specific multiplier 
axx_list[0:47] = [{'axx_mult' : 'mul8s_1L2H', 'axx_power' : 0.8871, 'quant_bits' : 8, 'fake_quant' : False}] * 47

returned_power = [0]
replace_linear_layers(model,  AdaPT_Linear, axx_list, total_macs, total_params, layer_count=[0], returned_power = returned_power, initialize = False)  
print('Power of approximated operations: ', round(returned_power[0], 2), '%')
print('Model compiled.')

criterion = torch.nn.CrossEntropyLoss().to(device)
# Run evaluation on the validation dataset
top1 = evaluate_imagenet(model, val_data, criterion, print_freq=1000, device = device)

Power of approximated operations:  87.73 %
Model compiled.
Starting evaluation...


100%|██████████| 391/391 [1:10:10<00:00, 10.77s/it]

 * Acc@1 2.752 Acc@5 7.932


## 8. Optional retraining

Only use this if `calib_data` has corrected labels. The loader cell above prints whether corrected labels were used.


In [15]:
from classification.train import train_one_epoch
import torch

criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)

train_one_epoch(model, criterion, optimizer, calib_data, device, 0, 10)


Epoch: [0]  [ 0/79]  eta: 0:13:26  lr: 1e-06  img/s: 15.313784347814082  loss: 0.6344 (0.6344)  acc1: 83.5938 (83.5938)  acc5: 96.0938 (96.0938)  time: 10.2083  data: 1.8497  max mem: 8226
Epoch: [0]  [10/79]  eta: 0:09:45  lr: 1e-06  img/s: 15.415369729493412  loss: 0.7789 (0.7509)  acc1: 80.4688 (80.2557)  acc5: 96.0938 (95.5256)  time: 8.4821  data: 0.1685  max mem: 8397
Epoch: [0]  [20/79]  eta: 0:08:15  lr: 1e-06  img/s: 15.41303522504308  loss: 0.7061 (0.7477)  acc1: 79.6875 (79.6875)  acc5: 96.0938 (95.5729)  time: 8.3062  data: 0.0003  max mem: 8397
Epoch: [0]  [30/79]  eta: 0:06:49  lr: 1e-06  img/s: 15.411452147475005  loss: 0.6005 (0.7006)  acc1: 82.0312 (81.3508)  acc5: 96.0938 (95.7157)  time: 8.3023  data: 0.0003  max mem: 8397
Epoch: [0]  [40/79]  eta: 0:05:25  lr: 1e-06  img/s: 15.418989941414429  loss: 0.7426 (0.7359)  acc1: 78.9062 (80.4497)  acc5: 95.3125 (95.4649)  time: 8.3014  data: 0.0003  max mem: 8397
Epoch: [0]  [50/79]  eta: 0:04:01  lr: 1e-06  img/s: 15.4233

## 9. Re-run evaluation after retraining

In [14]:
top1, top5 = evaluate_imagenet(model, val_data, criterion, print_freq=1000, device = device)

Starting evaluation...


100%|██████████| 391/391 [1:10:12<00:00, 10.77s/it]

 * Acc@1 2.752 Acc@5 7.932


In [15]:
print(type(val_data.dataset))
print(val_data.dataset.target_transform)
print(val_data.dataset.classes[:5])
print(val_data.dataset[0][1])

<class 'torchvision.datasets.folder.ImageFolder'>
<function make_corrected_imagenet_loaders.<locals>.<lambda> at 0x7f7f7d89d120>
['0', '1', '10', '100', '101']
278































































## 11. GPU/memory diagnostics

In [11]:
!nvidia-smi


Fri Jun 12 00:59:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A16                     Off |   00000000:1B:00.0 Off |                    0 |
|  0%   36C    P0             25W /   62W |    1190MiB /  15356MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

pid, process_name, used_gpu_memory [MiB]
500859, [Not Found], 3048 MiB
514592, [Not Found], 6836 MiB
500859, [Not Found], 12608 MiB


In [7]:
import torch

print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
print("Reserved :", torch.cuda.memory_reserved()/1024**3, "GB")

Allocated: 0.0 GB
Reserved : 0.0 GB
